In [ ]:
# Install PyTorch and related packages
# - torch: main deep learning library (used by CLIP)
# - torchvision: utilities for image processing
# - torchaudio: audio utilities (not strictly needed here, but useful if you extend later)
!pip install -q torch torchvision torchaudio

# Install OpenAI's CLIP model directly from GitHub
# CLIP is used to understand what's happening in video frames (e.g., celebrations, crowd, etc.)
!pip install -q git+https://github.com/openai/CLIP.git

# Install computer vision and plotting tools
# - opencv-python: read video frames and process images
# - pillow: image handling (used by CLIP preprocessing)
# - ffmpeg-python: interface to ffmpeg for video processing
# - matplotlib: plotting (e.g., excitement over time)
!pip install -q opencv-python pillow ffmpeg-python matplotlib


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00


In [15]:
!pip install -q git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00


In [16]:
# Import the CLIP library (installed from GitHub)
# This gives access to the CLIP model for vision-language understanding
import clip

# Simple check to confirm that CLIP was imported successfully
# If this prints without errors, the installation worked
print("CLIP imported OK")


CLIP imported OK


In [2]:
# Import the Google Drive integration for Colab
# This allows Colab to access files stored in your Google Drive
from google.colab import drive

# Mount (attach) your Google Drive to the Colab file system
# After this, your Drive will be accessible under the folder: /content/drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# Path to the input video file on Google Drive
# /content/drive/MyDrive/ is the root of your Google Drive in Colab
# "highlights" is the folder you created in Drive
# "IMG_1596.MOV" is the actual video file you want to process
VIDEO_PATH = "/content/drive/MyDrive/highlights/IMG_1596.MOV"

In [4]:
# List all files inside the "highlights" folder on your Google Drive
# This helps you verify that your video file is really there
!ls /content/drive/MyDrive/highlights


IMG_1596.MOV


In [6]:
# Remove the "segments" folder and everything inside it (if it already exists)
# This ensures we start fresh and don't mix old video segments with new ones
!rm -rf segments

# Create a new empty "segments" folder
# This is where ffmpeg will save the video chunks (out0000.mp4, out0001.mp4, etc.)
!mkdir -p segments


In [7]:
!ls

drive  sample_data  segments


In [8]:
# ffmpeg: command-line tool to process video
# -i "$VIDEO_PATH"        -> input video file (your .MOV from Google Drive)
# -map 0:v                -> select only the video stream (ignore audio)
# -c:v libx264            -> re-encode video using H.264 (OpenCV-friendly)
# -pix_fmt yuv420p       -> convert to 8-bit pixel format for compatibility
# -crf 23                 -> quality setting (lower = better quality, bigger files)
# -preset veryfast       -> encoding speed/quality tradeoff (faster encoding)
# -f segment              -> tell ffmpeg to split output into segments
# -segment_time 5         -> each segment is 5 seconds long
# -reset_timestamps 1    -> reset timestamps for each segment
# segments/out%04d.mp4   -> output file pattern (out0000.mp4, out0001.mp4, ...)

!ffmpeg -i "$VIDEO_PATH" -map 0:v -c:v libx264 -pix_fmt yuv420p -crf 23 -preset veryfast \
-f segment -segment_time 5 -reset_timestamps 1 segments/out%04d.mp4


Streaming output truncated to the last 5000 lines.
[hevc @ 0x579f4ae63540] Skipping NAL unit 62
[hevc @ 0x579f4aeee4c0] Skipping NAL unit 62
[hevc @ 0x579f4ae4af80] Skipping NAL unit 62
[hevc @ 0x579f4ae63540] Skipping NAL unit 62
[hevc @ 0x579f4aeee4c0] Skipping NAL unit 62
[hevc @ 0x579f4ae4af80] Skipping NAL unit 62
[hevc @ 0x579f4ae63540] Skipping NAL unit 62
[hevc @ 0x579f4aeee4c0] Skipping NAL unit 62
[hevc @ 0x579f4ae4af80] Skipping NAL unit 62
[hevc @ 0x579f4ae63540] Skipping NAL unit 62
[hevc @ 0x579f4aeee4c0] Skipping NAL unit 62
[hevc @ 0x579f4ae4af80] Skipping NAL unit 62
[hevc @ 0x579f4ae63540] Skipping NAL unit 62
[hevc @ 0x579f4aeee4c0] Skipping NAL unit 62
[hevc @ 0x579f4ae4af80] Skipping NAL unit 62
[hevc @ 0x579f4ae63540] Skipping NAL unit 62
[hevc @ 0x579f4aeee4c0] Skipping NAL unit 62
[hevc @ 0x579f4ae4af80] Skipping NAL unit 62
[hevc @ 0x579f4ae63540] Skipping NAL unit 62
[hevc @ 0x579f4aeee4c0] Skipping NAL unit 62
[hevc @ 0x579f4ae4af80] Skipping NAL unit 62
[hev

In [9]:
!ls segments | head


out0000.mp4
out0001.mp4
out0002.mp4
out0003.mp4
out0004.mp4
out0005.mp4
out0006.mp4
out0007.mp4
out0008.mp4
out0009.mp4


In [10]:
# Use ffmpeg to inspect a video file and print its metadata/streams
# This does NOT modify the file — it just shows info in the output
# You can use this to check:
#   - the codec (H.264, etc.)
#   - resolution (width x height)
#   - frame rate (fps)
#   - whether the file has video/audio streams
!ffmpeg -i segments/out0000.mp4


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [12]:
import os  # Standard Python library for interacting with the file system

# Get a sorted list of all files inside the "segments" folder
# This will be your list of video chunks: out0000.mp4, out0001.mp4, ...
segment_files = sorted(os.listdir("segments"))

# Print:
# 1) The total number of segments created
# 2) The first 5 filenames (just to sanity-check the naming/order)
print(len(segment_files), segment_files[:5])


72 ['out0000.mp4', 'out0001.mp4', 'out0002.mp4', 'out0003.mp4', 'out0004.mp4']


In [13]:
# List the files inside the "segments" folder
# The "| head" part means: only show the first few entries
# This is useful to quickly check that the video segments were created correctly
# without flooding the output with hundreds of filenames
!ls segments | head


out0000.mp4
out0001.mp4
out0002.mp4
out0003.mp4
out0004.mp4
out0005.mp4
out0006.mp4
out0007.mp4
out0008.mp4
out0009.mp4


Main code


In [ ]:
# Import required libraries
# os: file system operations (listing segment files)
# cv2: OpenCV for reading video frames
# torch: PyTorch (used by CLIP)
# clip: CLIP model for vision-language understanding
import os, cv2, torch, clip

# Numerical operations
import numpy as np

# Image handling (used by CLIP preprocessing)
from PIL import Image

# Plotting library (for visualization of excitement over time)
import matplotlib.pyplot as plt

# Select GPU if available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the CLIP model and preprocessing function
# ViT-B/32 is a good balance between speed and accuracy
clip_model, preprocess = clip.load("ViT-B/32", device=device)

# Text prompts that describe what we consider "highlights" in volleyball
# CLIP will compare video frames against these concepts
TEXT_PROMPTS = [
    "volleyball players celebrating a point",
    "volleyball team celebrating after a rally",
    "crowd cheering at a volleyball match",
    "players high-fiving after scoring a point in volleyball",
]

# Tokenize the text prompts so CLIP can process them
text_tokens = clip.tokenize(TEXT_PROMPTS).to(device)

def load_sampled_frames(video_path, num_frames=4):
    """
    Load a small number of evenly spaced frames from a video segment.
    This keeps memory and computation low while still capturing motion.
    """
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []

    # Pick evenly spaced frame indices
    idxs = np.linspace(0, total-1, num_frames).astype(int)
    targets = set(idxs.tolist())

    frames = []
    i = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if i in targets:
            frames.append(frame)
        i += 1

    cap.release()
    return frames

def clip_semantic_score(frame_bgr):
    """
    Compute how semantically similar a frame is to the volleyball highlight prompts
    using CLIP (e.g., celebrations, cheering, high-fives).
    """
    # Convert BGR (OpenCV format) to RGB for PIL/CLIP
    image = preprocess(Image.fromarray(frame_bgr[..., ::-1])).unsqueeze(0).to(device)
    with torch.no_grad():
        img_feat = clip_model.encode_image(image)
        txt_feat = clip_model.encode_text(text_tokens)
        # Cosine similarity between image and text embeddings
        sim = torch.cosine_similarity(img_feat, txt_feat).mean().item()
    return sim

def motion_score(frames):
    """
    Measure how much motion there is between sampled frames.
    High motion often corresponds to exciting rallies or celebrations.
    """
    if len(frames) < 2:
        return 0.0
    m = 0.0
    for i in range(1, len(frames)):
        diff = cv2.absdiff(frames[i-1], frames[i])
        m += np.mean(diff)
    return float(m / (len(frames)-1))

def scene_change_score(frames, threshold=25):
    """
    Count how many strong visual changes happen between frames.
    Broadcast cuts or sudden camera changes often follow big points.
    """
    if len(frames) < 2:
        return 0.0
    cuts = 0
    for i in range(1, len(frames)):
        diff = cv2.absdiff(frames[i-1], frames[i])
        if diff.mean() > threshold:
            cuts += 1
    return cuts

# List all segmented video files (e.g., out0000.mp4, out0001.mp4, ...)
segment_files = sorted(os.listdir("segments"))

# -------------------------
# Pass 1: Fast pre-filter
# -------------------------
# Quickly score segments using cheap features (motion + scene changes)
fast_scores = []
for f in segment_files:
    frames = load_sampled_frames(f"segments/{f}", num_frames=3)
    fast = motion_score(frames) + scene_change_score(frames)
    fast_scores.append(fast)

# Keep only the top 35% of segments for expensive CLIP processing
keep_n = max(1, int(0.35 * len(segment_files)))
keep_idx = set(np.argsort(fast_scores)[-keep_n:])

# -------------------------
# Pass 2: Full AI scoring
# -------------------------
# Combine CLIP semantics + motion + scene change into a single highlight score
scores = [0.0] * len(segment_files)
for i, f in enumerate(segment_files):
    if i not in keep_idx:
        continue
    frames = load_sampled_frames(f"segments/{f}", num_frames=4)
    if not frames:
        continue
    mid = frames[len(frames)//2]

    s_clip = clip_semantic_score(mid)
    s_motion = motion_score(frames)
    s_scene = scene_change_score(frames)

    # Weighted combination (tuned for volleyball)
    scores[i] = 0.55*s_clip + 0.30*(s_motion/50.0) + 0.15*(s_scene/3.0)

# -------------------------
# Visualization (optional)
# -------------------------
# Plot how "exciting" each segment is over time
plt.plot(scores)
plt.title("Visual Excitement Over Time (Volleyball)")
plt.xlabel("Segment")
plt.ylabel("Score")
plt.show()

# -------------------------
# Select top highlight segments
# -------------------------
top_k = 8  # number of highlight segments to include
idx = np.argsort(scores)[-top_k:]
idx = sorted(idx)

# Write selected segments to a file list for ffmpeg concatenation
with open("list.txt", "w") as f:
    for i in idx:
        f.write(f"file 'segments/{segment_files[i]}'\n")

print("Selected highlight segments:", idx)


In [20]:
# Use ffmpeg to concatenate (stitch together) multiple video segments into one video
# -f concat        -> use ffmpeg's "concat" demuxer (reads a list of files to merge)
# -safe 0          -> allow file paths outside the current directory (needed in Colab sometimes)
# -i list.txt      -> text file containing the list of segment files to merge
# -c copy          -> copy the video streams without re-encoding (fast, no quality loss)
# highlights.mp4   -> output file: your final highlights video
!ffmpeg -f concat -safe 0 -i list.txt -c copy highlights.mp4


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [21]:
# Import Colab utility for downloading files from the Colab environment to your computer
from google.colab import files

# Download the generated highlights video (highlights.mp4) to your local machine
# After running this, your browser will prompt you to save the file
files.download("highlights.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>